# GStore Revenue Prediction — demo

A thin walkthrough of the `garev` package on synthetic data. No 31 GB download
required. To run against the real competition data, replace the sample path with
`data/train_v2.csv`.

```bash
pip install -e ".[dev,explain]"
```

In [ ]:
from pathlib import Path

from garev.data.sample import write_sample
from garev.pipeline import load_sessions, make_timeframe, train, predict_submission
from garev.models.evaluate import evaluate, feature_importance

# 1. Synthetic, raw-shaped dataset
csv_path = write_sample(Path("../data/sample.csv"), n_visitors=2000)
csv_path

In [ ]:
# 2. Load + clean sessions, then build a forward-looking timeframe
sessions = load_sessions(csv_path)
dataset = make_timeframe(sessions)
print(f"{len(dataset.features):,} visitors, {int(dataset.is_buyer.sum())} future buyers")
dataset.features.head()

In [ ]:
# 3. Train the two-stage model and inspect metrics
trained = train(dataset)
report = evaluate(trained.model, trained._align(dataset.features), dataset.is_buyer, dataset.log_revenue)
report.as_dict()

In [ ]:
# 4. Which features drive the return-buyer classifier?
feature_importance(trained.model, top_n=15)

In [ ]:
# 5. Score every visitor → Kaggle-format submission
submission = predict_submission(trained, sessions)
submission.sort_values("PredictedLogRevenue", ascending=False).head()